# RAG Q&A and Evaluation

This notebook builds a retrieval-augmented generation (RAG) pipeline on top of the movie knowledge corpus prepared in `01_rag_data_prep_and_indexing`. It covers three stages:

1. **Semantic Retrieval** — Load the embedding model and Delta table, then retrieve the most relevant chunks for a user query using cosine similarity.
2. **Prompt Engineering & Evaluation** — Build both a RAG-grounded prompt and a "blind" prompt (no context) to compare answer quality and highlight hallucination risk.
3. **Audit Logging** — Persist each query and its retrieved context to a Delta audit table for monitoring and governance.

## Step 1: Semantic Retrieval

Load the `all-MiniLM-L6-v2` embedding model and the `movie_knowledge_chunks` Delta table, then perform cosine-similarity retrieval to find the most relevant text chunks for a given query.

In [0]:
import numpy as np
from sentence_transformers import SentenceTransformer

# --- Unity Catalog configuration (must match the prep notebook) ---
CATALOG = "main"
SCHEMA = "lab_data"

# Load the embedding model and the persisted chunks from the Delta table
model = SentenceTransformer("all-MiniLM-L6-v2")
chunks_df = spark.table(f"{CATALOG}.{SCHEMA}.movie_knowledge_chunks").toPandas()

# --- Similarity utility ---
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve_relevant_chunks(query: str, top_k: int = 2, filter_movie: str = None) -> list:
    """Find the most semantically similar text chunks for a given user query."""
    query_vector = model.encode(query)
    
    candidates = chunks_df.copy()
    if filter_movie:
        candidates = candidates[candidates["movie_title"] == filter_movie]
        
    # Score each candidate chunk by cosine similarity to the query vector
    scores = []
    for _, row in candidates.iterrows():
        sim = cosine_similarity(query_vector, row["embedding"])
        scores.append((sim, row["content"], row["movie_title"]))
        
    # Sort by descending relevance and return the top-k results
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:top_k]

# --- Test the retrieval pipeline ---
test_query = "Why was the hospital explosion delayed in The Dark Knight?"
retrieved = retrieve_relevant_chunks(test_query, top_k=2)

print(f"🔍 Query: '{test_query}'\n")
for idx, (score, content, title) in enumerate(retrieved, 1):
    print(f"Top-{idx} [Score: {score:.4f}] [{title}]:\n{content}\n")

## Step 2: Prompt Engineering & Evaluation

Construct two prompt variants for the same question: a **RAG prompt** that injects retrieved context (grounded) and a **blind prompt** with no context (hallucination-prone). This demonstrates how retrieval grounding improves answer accuracy.

In [0]:
def format_rag_prompt(question: str, retrieved_chunks: list) -> str:
    """Build a system prompt that injects the retrieved context to ground the LLM's answer."""
    context_text = "\n".join([f"- {chunk[1]}" for chunk in retrieved_chunks])
    return f"""You are a movie knowledge assistant. Answer the question based ONLY on the provided context. If you don't know, say "I don't know based on the context".

CONTEXT:
{context_text}

QUESTION: {question}
ANSWER:"""

def format_blind_prompt(question: str) -> str:
    """Build a prompt without any context (intentionally provokes hallucinations)."""
    return f"Answer the following question: {question}"

# --- Evaluation Setup ---
test_question = "Why was the hospital explosion delayed in The Dark Knight?"
retrieved_data = retrieve_relevant_chunks(test_question, top_k=1)

rag_prompt = format_rag_prompt(test_question, retrieved_data)
blind_prompt = format_blind_prompt(test_question)

print("=== ❌ BLIND PROMPT (Without RAG) ===")
print(blind_prompt)
print("\n[Expected LLM Hallucination / Risk]: The model might guess it was a technical error with the pyrotechnics, missing the acting nuance.")

print("\n\n=== ✅ RAG PROMPT (With Vector Retrieval) ===")
print(rag_prompt)
print("\n[Expected RAG Result]: The model will accurately state that Ledger remained in character and played with the detonator until it resumed, because this exact fact is injected into the context.")

## Step 3: Audit Logging

Log the test query and its retrieved context to a Unity Catalog Delta table for monitoring, governance, and traceability. Each query gets a unique ID and a UTC timestamp.

In [0]:
import uuid
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from datetime import datetime, timezone

# --- Define the audit log schema ---
audit_schema = StructType([
    StructField("query_id", StringType(), False),
    StructField("user_query", StringType(), False),
    StructField("retrieved_context", StringType(), True),
    StructField("answer", StringType(), True),
    StructField("timestamp", TimestampType(), False)
])

# --- Log the test query, retrieved context, and RAG-grounded answer ---
audit_log = [{
    "query_id": str(uuid.uuid4()),
    "user_query": test_question,
    "retrieved_context": retrieved_data[0][1] if retrieved_data else "None",
    "answer": rag_prompt,
    "timestamp": datetime.now(timezone.utc)
}]

audit_df = spark.createDataFrame(audit_log, schema=audit_schema)
audit_table_name = f"{CATALOG}.{SCHEMA}.rag_audit_log"

# Write the audit log entry to the Delta table (append mode)
audit_df.write.format("delta").mode("append").saveAsTable(audit_table_name)
print(f"✅ Query successfully logged to {audit_table_name} for monitoring & governance.")

# --- Display the last added row to confirm successful auditing ---
print("\n📋 Last audit log entry:")
display(spark.sql(f"SELECT * FROM {audit_table_name} ORDER BY timestamp DESC LIMIT 1"))